In [1]:

from google.colab import drive
from pathlib import Path

mount = Path("/content/gdrive")

if not (mount / "MyDrive").is_dir():
    drive.mount(str(mount))

root = mount / "MyDrive/EdgeTB_Project"
work = root / "tbx11k_training_study"
ablation = work / "ablation_study"

model_path = (
    ablation / "models/mobilenetv2_full_seed42.keras"
)

train_cache = ablation / "cache/train_full.npy"
val_cache = ablation / "cache/val_full.npy"

output_dir = ablation / "compression_study"
output_dir.mkdir(parents=True, exist_ok=True)

for name, path in {
    "Keras model": model_path,
    "Training images": train_cache,
    "Validation images": val_cache
}.items():
    assert path.is_file(), f"Missing: {path}"
    print(name, "Found")

print("Output:", output_dir)


Mounted at /content/gdrive
Keras model Found
Training images Found
Validation images Found
Output: /content/gdrive/MyDrive/EdgeTB_Project/tbx11k_training_study/ablation_study/compression_study


Compress the model

In [2]:

import numpy as np
import pandas as pd
import tensorflow as tf

model = tf.keras.models.load_model(
    str(model_path),
    compile=False
)

baseline_path = (
    ablation / "edge_deployment/mobilenetv2_full_fp32.tflite"
)

assert baseline_path.is_file()

train_images = np.load(
    train_cache,
    mmap_mode="r"
)

rng = np.random.default_rng(42)
calibration_ids = rng.choice(
    len(train_images),
    size=200,
    replace=False
)

def representative_data():
    for idx in calibration_ids:
        gray = np.asarray(
            train_images[int(idx)],
            dtype=np.float32
        ) / 255.0

        rgb = np.repeat(
            gray[..., None],
            3,
            axis=-1
        )

        yield [rgb[None, ...]]

def convert_model(mode):
    converter = tf.lite.TFLiteConverter.from_keras_model(model)
    converter.optimizations = [tf.lite.Optimize.DEFAULT]

    if mode == "fp16":
        converter.target_spec.supported_types = [tf.float16]

    elif mode == "int8":
        converter.representative_dataset = representative_data
        converter.target_spec.supported_ops = [
            tf.lite.OpsSet.TFLITE_BUILTINS_INT8
        ]
        converter.inference_input_type = tf.float32
        converter.inference_output_type = tf.float32

    return converter.convert()

paths = {"fp32": baseline_path}

for mode in ["fp16", "int8"]:
    print(f"\nConverting {mode}")

    try:
        compressed = convert_model(mode)

        path = output_dir / f"mobilenetv2_full_{mode}.tflite"
        path.write_bytes(compressed)

        paths[mode] = path
        print("Saved:", path)

    except Exception as exc:
        print(f"{mode} conversion failed:")
        print(type(exc).__name__, str(exc)[:1000])

rows = []

for mode, path in paths.items():
    rows.append({
        "Variant": mode,
        "Size MB": path.stat().st_size / (1024 ** 2),
        "Reduction %": (
            1 - path.stat().st_size / baseline_path.stat().st_size
        ) * 100
    })

sizes = pd.DataFrame(rows)
sizes.to_csv(output_dir / "compression_sizes.csv", index=False)

print("\nModel sizes")
print(sizes.round(3).to_string(index=False))



Converting fp16
Saved artifact at '/tmp/tmpu6v6c8hn'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 224, 224, 3), dtype=tf.float32, name='input_layer_1')
Output Type:
  TensorSpec(shape=(None, 1), dtype=tf.float32, name=None)
Captures:
  132566981340304: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132566981338960: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132566981340880: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132566981341648: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132566981341072: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132566981341456: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132566981341264: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132566981342224: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132566981340112: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132566981342032: TensorSpec(shape=(), dtype=tf.resource, name=None

/usr/local/lib/python3.13/dist-packages/tensorflow/lite/python/convert.py:846: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(


Saved: /content/gdrive/MyDrive/EdgeTB_Project/tbx11k_training_study/ablation_study/compression_study/mobilenetv2_full_int8.tflite

Model sizes
Variant  Size MB  Reduction %
   fp32    8.478        0.000
   fp16    4.298       49.310
   int8    2.605       69.274


Validate compressed models

In [4]:

import numpy as np
import pandas as pd
import tensorflow as tf

from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    confusion_matrix
)

manifest = pd.read_csv(
    work / "manifests/tbx11k_val.csv"
)

comparison = pd.read_csv(
    ablation / "results/segmentation_ablation_comparison.csv"
)

row = comparison[
    (comparison["Model"] == "Full X-ray") &
    (
        comparison["Threshold strategy"] ==
        "Validation 90% sensitivity"
    )
]

assert len(row) == 1

threshold = float(row.iloc[0]["Threshold"])
labels = manifest["label"].to_numpy(dtype=int)
images = np.load(val_cache, mmap_mode="r")

def predict_lite(path):
    interpreter = tf.lite.Interpreter(
        model_path=str(path),
        num_threads=2
    )
    interpreter.allocate_tensors()

    input_info = interpreter.get_input_details()[0]
    output_info = interpreter.get_output_details()[0]

    assert input_info["dtype"] == np.float32
    assert output_info["dtype"] == np.float32
    assert input_info["shape"].tolist() == [1, 224, 224, 3]

    scores = np.empty(len(images), dtype=np.float32)

    for i in range(len(images)):
        gray = np.asarray(
            images[i], dtype=np.float32
        ) / 255.0

        batch = np.repeat(
            gray[..., None], 3, axis=-1
        )[None, ...]

        interpreter.set_tensor(input_info["index"], batch)
        interpreter.invoke()

        scores[i] = interpreter.get_tensor(
            output_info["index"]
        ).reshape(-1)[0]

    return scores

scores_by_mode = {}
records = []

for mode, path in paths.items():
    print(f"Evaluating {mode}")

    scores = predict_lite(path)
    scores_by_mode[mode] = scores

    predicted = (scores >= threshold).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        labels,
        predicted,
        labels=[0, 1]
    ).ravel()

    records.append({
        "Variant": mode,
        "Size MB": path.stat().st_size / 1024**2,
        "ROC-AUC": roc_auc_score(labels, scores),
        "Average precision": average_precision_score(
            labels, scores
        ),
        "Sensitivity": tp / (tp + fn),
        "Specificity": tn / (tn + fp),
        "False negatives": int(fn),
        "False positives": int(fp),
        "Max difference from FP32": float(
            np.max(np.abs(scores - scores_by_mode["fp32"]))
        )
    })

summary = pd.DataFrame(records)

summary.to_csv(
    output_dir / "compression_validation.csv",
    index=False
)

print("\nCompression validation")
print(
    summary.to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}"
    )
)

print("\nThreshold used:", threshold)


Evaluating fp32


/usr/local/lib/python3.13/dist-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


Evaluating fp16


/usr/local/lib/python3.13/dist-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


Evaluating int8


/usr/local/lib/python3.13/dist-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)



Compression validation
Variant  Size MB  ROC-AUC  Average precision  Sensitivity  Specificity  False negatives  False positives  Max difference from FP32
   fp32 8.478134 0.957794           0.843881     0.895000     0.928125               21              115                  0.000000
   fp16 4.297596 0.957447           0.842220     0.895000     0.923750               21              122                  0.049387
   int8 2.604950 0.953847           0.829511     0.890000     0.923125               22              123                  0.412617

Threshold used: 0.3904514014720917


In [5]:

import time
import numpy as np
import pandas as pd
import tensorflow as tf

rng = np.random.default_rng(42)

sample_ids = rng.choice(
    len(images),
    size=50,
    replace=False
)

prepared = []

for idx in sample_ids:
    gray = np.asarray(
        images[int(idx)],
        dtype=np.float32
    ) / 255.0

    batch = np.repeat(
        gray[..., None],
        3,
        axis=-1
    )[None, ...]

    prepared.append(batch)

interpreters = {}

for mode, path in paths.items():
    interpreter = tf.lite.Interpreter(
        model_path=str(path),
        num_threads=2
    )

    interpreter.allocate_tensors()

    input_info = interpreter.get_input_details()[0]

    assert input_info["shape"].tolist() == [1, 224, 224, 3]
    assert input_info["dtype"] == np.float32

    interpreters[mode] = {
        "interpreter": interpreter,
        "input_id": input_info["index"]
    }

for mode, item in interpreters.items():
    for batch in prepared[:15]:
        item["interpreter"].set_tensor(
            item["input_id"], batch
        )
        item["interpreter"].invoke()

records = []

for round_number in range(1, 4):
    # Interleave models for each image to reduce order bias.
    for image_index, batch in enumerate(prepared):
        order = rng.permutation(list(paths.keys()))

        for mode in order:
            item = interpreters[mode]
            interpreter = item["interpreter"]

            interpreter.set_tensor(
                item["input_id"], batch
            )

            start = time.perf_counter()
            interpreter.invoke()
            elapsed_ms = (
                time.perf_counter() - start
            ) * 1000

            records.append({
                "Variant": mode,
                "Round": round_number,
                "Image": image_index,
                "Inference ms": elapsed_ms
            })

    print(f"Round {round_number}/3 completed")

timings = pd.DataFrame(records)

summary = timings.groupby("Variant")["Inference ms"].agg(
    Mean_ms="mean",
    Median_ms="median",
    SD_ms="std",
    P95_ms=lambda x: np.percentile(x, 95)
).reset_index()

summary["Size MB"] = summary["Variant"].map({
    mode: path.stat().st_size / 1024**2
    for mode, path in paths.items()
})

timings.to_csv(
    output_dir / "compression_cpu_timings.csv",
    index=False
)

summary.to_csv(
    output_dir / "compression_cpu_summary.csv",
    index=False
)

print("\nCompression CPU benchmark")
print(summary.round(3).to_string(index=False))


/usr/local/lib/python3.13/dist-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


Round 1/3 completed
Round 2/3 completed
Round 3/3 completed

Compression CPU benchmark
Variant  Mean_ms  Median_ms  SD_ms  P95_ms  Size MB
   fp16   16.418     17.038  7.380  30.691    4.298
   fp32   16.600     17.182  7.420  32.406    8.478
   int8   21.476     22.892  8.396  38.420    2.605
